# Named entity recognition (NER)

In [1]:
# !pip install yargy stanza transformers -q

In [ ]:
import stanza
stanza.download('ru')

## NER План

* Постановка задачи и примеры
* Данные
* Оценка качества
* Подходы к решению

## Постановка задачи и примеры

### Базовая постановка

Есть текст, мы хотим выделить в нем все именованые сущности (ИС). _Что такое именованые сущности? Что значит выделить?_
В данном случае именованые сущности - это чаще всего любые имена собственные. Есть более сложная задача, когда мы не привязываемся к именованности, а выделяем сущности в целом, но тут начинаются сложности с определениями.
Выделить - определить, из каких токенов текста состоит именованая сущность. 

![](https://github.com/named-entity/hse-nlp/blob/master/4th_year/Slides/3_NER_img/Raul_castro_example.png?raw=true)

[Источник](http://opencorpora.org/wiki/Nermanual/2/model)

Данная задача является частью большого блока задач по извлечению информации из текста, куда входят еще:
* извлечение отношений: часть-целое, персона-статус/должность/фамилия/место работы… (Relationship Extraction & Classification)
* извлечение фактов/событий: продажа, арест, стихийные бедствия… - факт + участники факта + локализация + время (Event Detection & Classification)
* извлечение речи персон
* временной анализ событий (Temporal Analysis)
* заполнение шаблонов о событии/факте

### Типы именованных сущностей
Чаще всего мы хотим не просто выделить кусочки текста с ИС, а еще и определить их тип, так как географическое название и ФИО - это вещи достаточно разные, чтобы их имело смысл обрабатывать по-разному.

Стандартные типы:
- Person *Paris Hilton*
- Location *Paris*
- Organization *Hilton*

Более специфические:
- LocOrg (см. FactRuEval-2016) *конференция прошла в ВШЭ*
- валюта / сумма + валюта  *$500*
- даты *November 3rd*
- именованные события *World War II*

NB! Хороший пример, как сущность зависит от задачи

<img width=800 src="https://github.com/named-entity/hse-nlp/blob/master/4th_year/Slides/3_NER_img/twitter.png?raw=true"/>

### Способы решения
С точки зрения алгоритмов решения можно выделить три блока глобально различающихся подходов:
1. Основанные на правилах и словарях
2. Основанные на классификации: классифицируем каждый токен на предмет принадлежности к ИС (и вариации на тему этой постановки) или получаем индексы начала и конца спана ИС
3. Генеративные: на основе текста как затравки хотим сгенерировать ИС и ее тип (редкий вариант для особо сложных случаев)

### Почему это сложно?

* Особенности орфографии: с большой буквы пишутся не только ИС (начало предложения во многих языках и все существительные в немецком), а иногда и они не пишутся (китайский)
* Многозначность: одна и та же ИС можнт значить разное (Волга - река и Волга - машина)
* Необходимость учитывать контекст
* Большой уровень разнообразия (имена могут быть почти любые как и названия компаний)
* Сложность получения чистых и разнообразных данных для обучения
* Вложенность ИС как отдельная сложность и задача

### Примеры
1. Информационный поиск: выделяем ИС и ищем по ним или отдельно через ИС проверяем, что нашли нужное
2. Knowledge extraction: автоматическое пополнение онтологий, баз данных / знаний
3. Вопросно-ответные системы: выделение ответа на вопрос в тексте (по базе данных, например)
4. Аналитика контента: аналитика по упоминаемости персоны, репутация ВУЗов

<img width=600 src="https://github.com/named-entity/hse-nlp/blob/master/4th_year/Slides/3_NER_img/%D0%B2%D1%83%D0%B7%D1%8B.png?raw=true"/>

[Источник](https://ineo.hse.ru/data/2012/04/20/1250125810/UniversResults_HSE_IRO_seminar_01-11-2011.pdf)

5. Помощь в работе эксперта: рубрикация и кластеризация текстов, подсветка ИС в новостных текстах

<img width=500 src="https://github.com/named-entity/hse-nlp/blob/master/4th_year/Slides/3_NER_img/lenta_NE.png?raw=true"/>

Источник: lenta.ru

<img width=800 src="https://github.com/named-entity/hse-nlp/blob/master/4th_year/Slides/3_NER_img/eventos.png?raw=true"/>

Источник: [проект Eventos](https://publications.hse.ru/articles/97529325)

## Данные

Стандарты NER: 

* Message understanding conference (MUC), Automatic content extraction (ACE) ...
* SemEval workshops
* Conference on Computational Natural Language Learning (CoNLL)

Особенности получения данных:
* нужен специальный датасет с разметкой, поэтому нужны разметчики (которые склонны ошибаться) и инструкция (какие теги использовать, как размечать краевые случаи; и ее все равно никто не читает...)
* зависит от препроцессинга: в разных подходах границы токенов могут быть очень разные, нужно уметь маппить разметку с текущим решением
* готовые датасеты часто не сбалансированные с точки зрения домена и типов сущностей

### Разметка BIOES

(иногда - *BIO-tagging*)

**B** - beginning - 1 токен сущности

**I** - inside - не 1/последний токен сущности

**O** - out - не сущность

**E** - ending - последний токен сущности

**S** - single - сущность из одного токена

|Рауль|Модесто|Кастро|Рус|родился|в|городке|Биран|в|1931|году|.|
|:-:|:-:|:-:|:-:|:-:|:-:|:-:|:-:|:-:|:-:|:-:|:-:|
|B-PER|I-PER|I-PER|E-PER|OUT|OUT|OUT|S-LOC|OUT|OUT|B-DATE|E-DATE|OUT|


Недостатки: 
* один токен - один тег
* не учитываем вложенность
* маппинг в исходный текст через токенизацию

### BIO + CoNLL

[Nerus](https://github.com/natasha/nerus): NER + corpus

<img src="https://github.com/named-entity/hse-nlp/blob/master/4th_year/Slides/3_NER_img/nerus.png?raw=true"/>


### Датасеты для русского языка

* [Ru-Eval](https://ru-eval.github.io/resources.html), [RuEval-2014](http://www.dialog-21.ru/evaluation/2014/anaphora/)
* [FactRuEval-2016](https://github.com/dialogue-evaluation/factRuEval-2016)
* [Nerus](https://github.com/natasha/nerus)
* некоторые другие - в проекте [Corus](https://github.com/natasha/corus):
  * Collection-5
  * [WikiNER](https://www.sciencedirect.com/science/article/pii/S0004370212000276?via%3Dihub):
      * полуавтоматическая разметка
      * выделяем заголовки-NE
      * ищем их в текстах
* [Hugging Face datasets](https://huggingface.co/datasets?language=language:ru&sort=trending&search=ner)

Маппинг в онтологию для русского http://www.dialog-21.ru/media/3433/sysoevaaandrianovia.pdf

## Оценка качества

[Вот тут](https://github.com/davidsbatista/NER-Evaluation) можно найти пример кода с оценками качества

Так как задача сложная, существует множество способов оценки качества, но в среднем они близки к способам оценик для классификации (precision / recall)

- token-level: у каждого токена есть свой класс (BIO + тип сущности)
- span-level: сравниваем выделенные спаны и эталонные
    - учитывать правильно выделенные спаны с неправильным тегом?
    - учитывать пересечение эталонных спанов и автоматически выделенных?
- Более комплексные системы оценки
    - [ACE](http://www.eng.utah.edu/~cs6961/papers/ACE-2008-description.pdf) (или [тут](http://www.lrec-conf.org/proceedings/lrec2004/pdf/5.pdf))
    - [MUC](http://www.aclweb.org/anthology/M93-1007)
    - [SemEeval 2013](https://aclanthology.org/S13-2056.pdf)

### Принципы оценки SemEval

**Тип совпадения**:
* *Correct (COR)* : полное совпадение;
* *Incorrect (INC)* : предсказание и разметка не совпадают;
* *Partial (PAR)* : предсказание и разметка в чем-то похожи, но не совпадают до конца;
* *Missing (MIS)* : для данного примера из разметки нет предсказания;
* *Spurius (SPU)* : для данного предсказания нет примера из разметки;

**Тип оценки:**
* Strict: точное совпадение границ и тега;
* Exact: точное совпадение границ без учета тега;
* Partial: частичное пересечение границ без учета тега;
* Type: частичное пересечение разметки (границ и тегов)

См. https://www.davidsbatista.net/blog/2018/05/09/Named_Entity_Evaluation/


<table class="tg">
  <tbody><tr>
    <th class="text-center" colspan="1"><span style="font-weight:bold">Scenario</span></th>
    <th class="text-center" colspan="2"><span style="font-weight:bold">Golden Standard</span></th>
    <th class="text-center" colspan="2"><span style="font-weight:bold">System Prediction</span></th>
    <th class="text-center" colspan="4"><span style="font-weight:bold">Evaluation Schema</span></th>
  </tr>
  <tr>
    <td></td>
    <td><span style="font-weight:bold">Entity Type</span></td>
    <td><span style="font-weight:bold">Surface String</span></td>    
    <td><span style="font-weight:bold">Entity Type</span></td>
    <td><span style="font-weight:bold">Surface String</span></td>    
    <td><span class="text-center" style="font-weight:bold">Type</span></td>
    <td><span class="text-center" style="font-weight:bold">Partial</span></td>
    <td><span class="text-center" style="font-weight:bold">Exact</span></td>
    <td><span class="text-center" style="font-weight:bold">Strict</span></td>
  </tr>
  <tr>
    <td>III</td>
    <td>brand</td>
    <td>TIKOSYN</td>
    <td></td>
    <td></td>
    <td>MIS</td>
    <td>MIS</td>
    <td>MIS</td>
    <td>MIS</td>
  </tr>
  <tr>
    <td>II</td>
    <td></td>
    <td></td>
    <td>brand</td>
    <td>healthy</td>
    <td>SPU</td>
    <td>SPU</td>
    <td>SPU</td>
    <td>SPU</td>
  </tr>
  <tr>
    <td>V</td>
    <td>drug</td>
    <td>warfarin</td>
    <td>drug</td>
    <td>of warfarin</td>
    <td>COR</td>
    <td>PAR</td>
    <td>INC</td>
    <td>INC</td>
  </tr>
  <tr>
    <td>IV</td>
    <td>drug</td>
    <td>propranolol</td>
    <td>brand</td>
    <td>propranolol</td>
    <td>INC</td>
    <td>COR</td>
    <td>COR</td>
    <td>INC</td>
  </tr>
  <tr>
    <td>I</td>
    <td>drug</td>
    <td>phenytoin</td>
    <td>drug</td>
    <td>phenytoin</td>
    <td>COR</td>
    <td>COR</td>
    <td>COR</td>
    <td>COR</td>
  </tr>
  <tr>
    <td>I</td>  
    <td>Drug</td>
    <td>theophylline</td>
    <td>drug</td>
    <td>theophylline</td>
    <td>COR</td>
    <td>COR</td>
    <td>COR</td>
    <td>COR</td>
  </tr>
  <tr>
    <td>VI</td>
    <td>group</td>
    <td>contraceptives</td>
    <td>drug</td>
    <td>oral contraceptives</td>
    <td>INC</td>
    <td>PAR</td>
    <td>INC</td>
    <td>INC</td>
  </tr>
</tbody></table>

### Метрики SemEval

Как оценить количество релевантных и предсказанных элементов? (знаменатель в формулах Precision&Recall)

$\text{TRUE} = COR + INC + PAR + MIS = TP + FN$

$\text{PRED} = COR + INC + PAR + SPU = TP + FP$


**Exact Match (strict + exact):**

$\text{Precision} = \frac{COR}{PRED} = \frac{TP}{TP+FP}$

$\text{Recall} = \frac{COR}{TRUE} = \frac{TP}{TP+FN}$

**Partial Match (partial + type):**

$\text{Precision} = \frac{COR\ +\ 0.5\ \times\ PAR}{PRED} = \frac{TP}{TP+FP}$

$\text{Recall} = \frac{COR\ +\ 0.5\ \times\ PAR}{TRUE} = \frac{COR}{PRED} = \frac{TP}{TP+FP}$

## Подходы

### Использование rule-based подходов

* тексты одной тематики
* чётко ограниченная предметная область
* легко формализуемые сущности
* мало неоднозначности
* стандартные тексты (напр., заявления одного типа)

#### Пример: Томита-парсер
См. https://github.com/yandex/tomita-parser/blob/master/docs/ru/README.md

**Ключевые слова**
```cpp
TAuxDicArticle "месяц"
{
    key = "январь" | "февраль" | "март" | "апрель" | "май" | "июнь" |
          "июль" | "август" |   "сентябрь" | "октябрь" | "ноябрь" | "декабрь"
}
TAuxDicArticle "день_недели"
{
    key = "понедельник" | "вторник" | "среда" | "четверг" | "пятница" | "суббота" | "воскресенье"
}
```


**Правила грамматики**
```cpp
DayOfWeek -> Noun<kwtype="день_недели">;
Day -> AnyWord<wff=/([1-2]?[0-9])|(3[0-1])/>;
Month -> Noun<kwtype="месяц">;
YearDescr -> "год" | "г. "
Year -> AnyWord<wff=/[1-2]?[0-9]{1,3}г?\.?/>;
Year -> Year YearDescr;

// "понедельник, 3 сентября 2012г."
Date -> DayOfWeek interp (Date.DayOfWeek) (Comma)
        Day interp (Date.Day) 
        Month interp (Date.Month)
        (Year interp (Date.Year)); 
```

#### Словари ключевых слов

словарь / газетир

Можно собирать полуавтоматически:
- semantic similarity
- bootstrapping: находим похожие вхождения по шаблонам
| | |
|-|-|
|“The city like New York” | “The city like X ” |
|“My name is John” | “My name is Y” |
|“Улица ак. Варги” | “Улица ак. Z” |

#### natasha/yargy

In [ ]:
from yargy import rule, Parser, or_, and_
from yargy.predicates import gte, lte, caseless, normalized, dictionary

In [3]:
DAY = and_(
    gte(1),  # greater or equal
    lte(31)  # less or equal
)
MONTH = and_(
    gte(1),
    lte(12)
)
YEAR = and_(
    gte(1),
    lte(2018)
)

MONTHS = {
    'январь',
    'февраль',
    'март',
    'апрель',
    'мая',
    'июнь',
    'июль',
    'август',
    'сентябрь',
    'октябрь',
    'ноябрь',
    'декабрь'
}

MONTH_NAME = dictionary(MONTHS)  # Нормальная форма слова in value

In [4]:
YEAR_WORDS = or_(
    rule(caseless('г'), '.'),  # a.lower() == b.lower()
    rule(normalized('год'))   # Нормальная форма слова == value
)

DATE =  rule(
    DAY,
    MONTH_NAME,
    YEAR,
    YEAR_WORDS.optional()
)
parser = Parser(DATE)

In [6]:
text = '''
8 января 2014 года, 15 июня 2001 г.,
31 февраля 2018
'''
for match in parser.findall(text):
    print(match, end='\n\n')

Match(tokens=[Token(value='8', span=[1, 2), type='INT'), MorphToken(value='января', span=[3, 9), type='RU', forms=[Form('январь', Grams(NOUN,gent,inan,masc,sing))]), Token(value='2014', span=[10, 14), type='INT'), MorphToken(value='года', span=[15, 19), type='RU', forms=[Form('год', Grams(NOUN,gent,inan,masc,sing)), Form('год', Grams(Infr,NOUN,accs,inan,masc,plur)), Form('год', Grams(Infr,NOUN,inan,masc,nomn,plur))])], span=[1, 19))

Match(tokens=[Token(value='15', span=[21, 23), type='INT'), MorphToken(value='июня', span=[24, 28), type='RU', forms=[Form('июнь', Grams(NOUN,gent,inan,masc,sing))]), Token(value='2001', span=[29, 33), type='INT'), MorphToken(value='г', span=[34, 35), type='RU', forms=[Form('г', Grams(Abbr,Fixd,NOUN,gent,inan,masc,sing)), Form('г', Grams(Abbr,Fixd,NOUN,inan,loct,masc,sing)), Form('г', Grams(Abbr,Fixd,NOUN,inan,masc,nomn,sing)), Form('г', Grams(Abbr,Fixd,NOUN,anim,masc,nomn,sing)), Form('г', Grams(Abbr,Fixd,NOUN,anim,gent,masc,sing)), Form('г', Grams(Abbr,F

См. ещё примеры https://habr.com/ru/post/349864/

И ещё https://github.com/natasha/yargy-examples

### ML подходы

#### Classification-based

* выделяем кандидатов в ИС: часто правилами, но вообще, любым способом
* feature extraction: правила, эмбеддинги или что-то еще
* классифицируем: является ли ИС и к какой группе относится

> какие недостатки?

#### Sequence tagging

POS-tagging, NER - задачи *разметки последовательности*

Classification vs. sequence tagging:

| | Classification | Sequence tagging |
|:-:|:-:|:-:|
| **Целевая переменная** | Класс для конкретной ИС | Последовательность классов для текста |
| **Выделение кандидатов** | Обязательный предварительный шаг | Не требуется (обычно) |

**Генеративные и дискриминативные модели**

Дискриминативные модели - привычные нам классификаторы, которые пытаются предсказывать наиболее вероятные значения таргета при условии признаков.

| Генеративные | Дискриминативные |
|:-:|:-:|
| Моделируем совместное распределение $p(y,x) = p(x|y) * p(y) $ | Моделируем условную вероятность $p(y|x)$ |
| Можно использовать для классификации и генерации | Можно использовать только для классификации |
| Обучение с учителем и без | Только обучение с учителем |
| Часто проигрывают на маленьких, но понятных данных |  |
| Простейший классификатор - наивный Байес <br> $$p(y,x)=p(y)*\prod_{k=1}^K{p(x_k|y)}$$ | Простейший классификатор - логистическая регрессия <br> $p(y|x)=\frac{1}{Z}\exp\{\lambda_y + \sum_{j=1}^K{\lambda_{y,j}x_j}\}$ <br> , где $Z=\sum_y{exp\{\lambda_y+\sum_{j=1}^K{\lambda_{y,j}x_j}\}}$ |
| Простейший теггинг последовательностей - Hidden Markov Model (HMM) <br> $$p(x,y)=\prod_{t=1}^T{p(y_t|y_{t-1})p(x_t|y_t)}$$ | Простейший теггинг последовательностей - linear CRF |

<img src="https://waksoft.susu.ru/wp-content/uploads/2021/07/generative-and-discriminative-models-1.png"/>

### Готовые решения

In [ ]:
text =  """
Сегодня Поварская улица соединяет Новый Арбат и Садовое кольцо. 
А когда-то здесь проходила Волоцкая торговая дорога на Великий Новгород. 
В 1570 году по ней из новгородского похода возвращался Иван Грозный с опричным войском. 
После разделения страны на земщину и опричнину эта местность вошла в состав последней — здесь селились приближенные к царю люди.
"""

#### Stanza NLP library

https://stanfordnlp.github.io/stanza/

In [ ]:
nlp = stanza.Pipeline(lang='ru', processors='tokenize,ner')

In [ ]:
doc = nlp(text)
for sent in doc.sentences:
    for ent in sent.ents:  # достаем NE
        print(ent)

#### Hugging Face models:

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import pipeline

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("Babelscape/wikineural-multilingual-ner")
model = AutoModelForTokenClassification.from_pretrained("Babelscape/wikineural-multilingual-ner")

nlp = pipeline("ner", model=model, tokenizer=tokenizer, grouped_entities=True)

In [ ]:
ner_results = nlp(text)
print(ner_results)

#### А еще
* [PullEnti](https://github.com/pullenti/pullenti-wrapper)
* [spaCy](https://spacy.io/)
* [AllenNLP](https://github.com/allenai/allennlp)
* [AdaptNLP](https://github.com/Novetta/adaptnlp)
* [natasha/slovnet](https://github.com/natasha/slovnet)
* [DeepPavlov](https://deeppavlov.ai/)
* [StanfordNLP](https://stanfordnlp.github.io/CoreNLP/) (+ [NLTK]())
* [Hugging Face models](https://huggingface.co/models?pipeline_tag=token-classification&language=ru&sort=trending&search=ner)